In [5]:
import os
import joblib
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

print("1. Loading MNIST dataset...")

1. Loading MNIST dataset...


In [6]:
mnist_cache = os.path.expanduser("~/.keras/datasets/mnist.npz")
if os.path.exists(mnist_cache):
    with np.load(mnist_cache) as data:
        X_train_raw, y_train_raw = data["x_train"], data["y_train"]
        X_test_raw, y_test_raw = data["x_test"], data["y_test"]
    X = np.concatenate([X_train_raw, X_test_raw]).reshape(-1, 784) / 255.0
    y = np.concatenate([y_train_raw, y_test_raw])
else:
    mnist = fetch_openml("mnist_784", version=1, as_frame=False)
    X = mnist.data / 255.0
    y = mnist.target.astype(int)

In [9]:
X_sub, _, y_sub, _ = train_test_split(
    X, y, train_size=20000, stratify=y, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42
)

print("2. Training Neural Network (MLP)...")
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    max_iter=50,
    random_state=42,
    verbose=True,
)
mlp.fit(X_train, y_train)

test_acc = mlp.score(X_test, y_test)
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")

2. Training Neural Network (MLP)...
Iteration 1, loss = 0.76161078
Iteration 2, loss = 0.27641085
Iteration 3, loss = 0.20498945
Iteration 4, loss = 0.16489739
Iteration 5, loss = 0.13368307
Iteration 6, loss = 0.10968378
Iteration 7, loss = 0.09199629
Iteration 8, loss = 0.07872026
Iteration 9, loss = 0.06183906
Iteration 10, loss = 0.05183318
Iteration 11, loss = 0.04536798
Iteration 12, loss = 0.03484169
Iteration 13, loss = 0.03049379
Iteration 14, loss = 0.02624486
Iteration 15, loss = 0.02069722
Iteration 16, loss = 0.01669502
Iteration 17, loss = 0.01259540
Iteration 18, loss = 0.01099600
Iteration 19, loss = 0.00894578
Iteration 20, loss = 0.00742718
Iteration 21, loss = 0.00617760
Iteration 22, loss = 0.00525805
Iteration 23, loss = 0.00479653
Iteration 24, loss = 0.00418342
Iteration 25, loss = 0.00334285
Iteration 26, loss = 0.00295317
Iteration 27, loss = 0.00268800
Iteration 28, loss = 0.00239453
Iteration 29, loss = 0.00221238
Iteration 30, loss = 0.00198723
Iteration 31,

In [10]:
joblib.dump(mlp, "digit_mlp_model.pkl")
print("Model saved successfully as 'digit_mlp_model.pkl'!")

Model saved successfully as 'digit_mlp_model.pkl'!


In [21]:
import cv2
import joblib
import numpy as np

In [22]:
# 1. Load trained model
model = joblib.load("digit_mlp_model.pkl")

In [25]:
# 2. Define preprocessing function
def preprocess_image(image_input):
    if isinstance(image_input, str):
        img = cv2.imread(image_input, cv2.IMREAD_GRAYSCALE)
    else:
        if len(image_input.shape) == 3:
            if image_input.shape[2] == 4:
                img = cv2.cvtColor(image_input, cv2.COLOR_RGBA2GRAY)
            else:
                img = cv2.cvtColor(image_input, cv2.COLOR_RGB2GRAY)
        else:
            img = image_input

# All these lines MUST be indented inside preprocess_image:
    img_resized = cv2.resize(img, (28, 28), interpolation=cv2.INTER_AREA)

    if np.mean(img_resized) > 127:
        img_resized = cv2.bitwise_not(img_resized)

    img_flattened = (img_resized.astype("float32") / 255.0).reshape(1, -1)
    return img_flattened, img_resized

In [26]:
# 3. Test on one sample from your dataset
sample_vector = X_test[0].reshape(1, -1)
pred_digit = model.predict(sample_vector)[0]
confidence = np.max(model.predict_proba(sample_vector)) * 100

print(f"Predicted Digit : {pred_digit}")
print(f"Confidence      : {confidence:.2f}%")
print(f"Actual Label    : {y_test[0]}")

Predicted Digit : 2
Confidence      : 100.00%
Actual Label    : 2


In [43]:
%%writefile app.py
import streamlit as st
import numpy as np
import cv2
import joblib
from streamlit_drawable_canvas import st_canvas
import pandas as pd
from PIL import Image

# ----------------- MODERN THEME & LAYOUT CONFIG -----------------
st.set_page_config(
    page_title="Handwritten Digit Intelligence",
    page_icon="⚡",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom High-End Styling
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;600;700;800&display=swap');

    html, body, [class*="css"] {
        font-family: 'Plus Jakarta Sans', sans-serif;
    }

    /* Gradient Hero Banner */
    .hero-container {
        background: linear-gradient(135deg, rgba(20, 24, 40, 0.95) 0%, rgba(10, 12, 20, 0.98) 100%);
        border: 1px solid rgba(255, 255, 255, 0.08);
        border-radius: 16px;
        padding: 2rem 2.5rem;
        margin-bottom: 2rem;
        box-shadow: 0 20px 40px -15px rgba(0,0,0,0.5);
    }
    
    .hero-title {
        font-size: 2.2rem;
        font-weight: 800;
        background: linear-gradient(90deg, #60A5FA 0%, #A78BFA 50%, #F472B6 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 0.4rem;
    }
    
    .hero-sub {
        color: #94A3B8;
        font-size: 1.05rem;
        margin-bottom: 0;
    }

    /* Glass Cards */
    .glass-card {
        background: rgba(30, 41, 59, 0.5);
        backdrop-filter: blur(12px);
        border: 1px solid rgba(255, 255, 255, 0.08);
        border-radius: 14px;
        padding: 1.5rem;
        box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.3);
    }

    /* Stat Badges */
    .stat-box {
        background: rgba(15, 23, 42, 0.7);
        border: 1px solid rgba(255, 255, 255, 0.06);
        border-radius: 12px;
        padding: 1.25rem;
        text-align: center;
        transition: transform 0.2s ease, border-color 0.2s ease;
    }
    .stat-box:hover {
        border-color: rgba(96, 165, 250, 0.4);
        transform: translateY(-2px);
    }
    .stat-num {
        font-size: 3rem;
        font-weight: 800;
        color: #38BDF8;
        line-height: 1;
        margin-bottom: 0.4rem;
    }
    .stat-label {
        font-size: 0.85rem;
        text-transform: uppercase;
        letter-spacing: 0.05em;
        color: #94A3B8;
        font-weight: 600;
    }

    /* Sidebar Refinements */
    [data-testid="stSidebar"] {
        background-color: #0b0f19;
        border-right: 1px solid rgba(255, 255, 255, 0.06);
    }
</style>
""", unsafe_allow_html=True)

# ----------------- BACKEND PREPROCESSING ENGINE -----------------
@st.cache_resource
def load_trained_model():
    return joblib.load("digit_mlp_model.pkl")

try:
    model = load_trained_model()
except Exception as e:
    st.error(f"Error loading inference weights: {e}")
    st.stop()

def center_digit_mass(img_20x20):
    m = cv2.moments(img_20x20)
    if m["m00"] == 0:
        return np.pad(img_20x20, ((4, 4), (4, 4)), mode='constant')

    cx = m["m10"] / m["m00"]
    cy = m["m01"] / m["m00"]

    shift_x = np.round(14.0 - (cx + 4)).astype(int)
    shift_y = np.round(14.0 - (cy + 4)).astype(int)

    padded = np.pad(img_20x20, ((4, 4), (4, 4)), mode='constant')
    M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    return cv2.warpAffine(padded, M, (28, 28), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

def extract_features(raw_image):
    if len(raw_image.shape) == 3:
        if raw_image.shape[2] == 4:
            gray = cv2.cvtColor(raw_image, cv2.COLOR_RGBA2GRAY)
        else:
            gray = cv2.cvtColor(raw_image, cv2.COLOR_RGB2GRAY)
    else:
        gray = raw_image

    if np.mean(gray) > 127:
        gray = cv2.bitwise_not(gray)

    coords = cv2.findNonZero(gray)
    if coords is None:
        return None, None

    x, y, w, h = cv2.boundingRect(coords)
    cropped = gray[y:y+h, x:x+w]

    if h > w:
        new_h = 20
        new_w = max(1, int(round((w * 20.0) / h)))
    else:
        new_w = 20
        new_h = max(1, int(round((h * 20.0) / w)))

    scaled_20x20 = cv2.resize(cropped, (new_w, new_h), interpolation=cv2.INTER_AREA)

    container = np.zeros((20, 20), dtype=np.uint8)
    y_off = (20 - new_h) // 2
    x_off = (20 - new_w) // 2
    container[y_off:y_off+new_h, x_off:x_off+new_w] = scaled_20x20

    final_28x28 = center_digit_mass(container)
    vector_784 = (final_28x28.astype("float32") / 255.0).reshape(1, -1)
    return vector_784, final_28x28

# ----------------- SIDEBAR PARAMETERS -----------------
with st.sidebar:
    st.markdown("###  Architecture Control")
    stroke_width = st.slider("Pen Calligraphy Weight", min_value=12, max_value=32, value=22, step=2)
    

# ----------------- HERO BANNER -----------------
st.markdown("""
<div class="hero-container">
    <div class="hero-title">Handwritten Digit Intelligence</div>
    <p class="hero-sub">Deep feature extraction and probability calibration for handwritten character intelligence.</p>
</div>
""", unsafe_allow_html=True)

# ----------------- MAIN STUDIO -----------------
left_panel, right_panel = st.columns([1.1, 1.3], gap="large")

eval_image = None
run_inference = False

with left_panel:
    st.markdown("#### Input Studio")
    studio_tabs = st.tabs([" Interactive Slate", " Image Upload"])
    
    with studio_tabs[0]:
        canvas_result = st_canvas(
            fill_color="#000000",
            stroke_width=stroke_width,
            stroke_color="#FFFFFF",
            background_color="#000000",
            height=300,
            width=300,
            drawing_mode="freedraw",
            update_streamlit=True,
            return_image_data=True,
            key="pro_canvas",
        )
        if st.button("Analyze Sketch", type="primary", use_container_width=True):
            if canvas_result is not None and canvas_result.image_data is not None:
                img_matrix = canvas_result.image_data.astype("uint8")
                if not np.all(img_matrix[:, :, :3] == 0):
                    eval_image = img_matrix
                    run_inference = True
                else:
                    st.warning("Slate is empty. Sketch a digit before processing.")

    with studio_tabs[1]:
        uploaded_doc = st.file_uploader("Select PNG or JPG digit scan", type=["png", "jpg", "jpeg"])
        if uploaded_doc is not None:
            bytes_data = np.asarray(bytearray(uploaded_doc.read()), dtype=np.uint8)
            decoded_doc = cv2.imdecode(bytes_data, cv2.IMREAD_UNCHANGED)
            st.image(decoded_doc, caption="Input Scan", width=160)
            if st.button("Analyze Uploaded Document", type="primary", use_container_width=True):
                eval_image = decoded_doc
                run_inference = True

with right_panel:
    st.markdown("#### Analytical Telemetry")
    
    if run_inference and eval_image is not None:
        vector, processed_28 = extract_features(eval_image)
        
        if vector is None:
            st.error("No stroke features recognized. Ensure drawing is clear and centered.")
        else:
            # Inference execution
            prob_dist = model.predict_proba(vector)[0]
            winner = int(np.argmax(prob_dist))
            confidence = float(prob_dist[winner] * 100)
            ranked = np.argsort(prob_dist)[::-1]

            # Top KPI Summary Cards
            col_kpi1, col_kpi2 = st.columns(2)
            with col_kpi1:
                st.markdown(f"""
                <div class="stat-box">
                    <div class="stat-num">{winner}</div>
                    <div class="stat-label">Identified Digit</div>
                </div>
                """, unsafe_allow_html=True)
            with col_kpi2:
                st.markdown(f"""
                <div class="stat-box">
                    <div class="stat-num" style="color: {'#34D399' if confidence > 75 else '#FBBF24'};">{confidence:.1f}%</div>
                    <div class="stat-label">Model Certainty</div>
                </div>
                """, unsafe_allow_html=True)

            st.write("")
            
            # Sub-Visualizations
            diag_col1, diag_col2 = st.columns([1, 1.8], gap="medium")
            
            with diag_col1:
                st.markdown("**Centroid Matrix (28×28)**")
                st.image(processed_28, width=130, clamp=True)
                st.caption("Barycentric moment alignment.")
                
            with diag_col2:
                st.markdown("**Top Alternative Candidates**")
                for rank in range(1, 4):
                    alt_digit = ranked[rank]
                    alt_prob = prob_dist[alt_digit] * 100
                    st.write(f"**Digit {alt_digit}** — `{alt_prob:.2f}%`")
                    st.progress(min(max(float(alt_prob / 100.0), 0.0), 1.0))

            st.write("")
            st.markdown("**Softmax Probability Spectrum Across All Classes (0–9)**")
            
            chart_df = pd.DataFrame({
                "Class": [str(d) for d in range(10)],
                "Probability (%)": [p * 100 for p in prob_dist]
            }).set_index("Class")
            
            st.bar_chart(chart_df, height=220, use_container_width=True)
    else:
        st.info("Draw a digit or upload a file on the left, then click **Analyze** to generate the real-time neural breakdown.")

Overwriting app.py
